### YOLOv8 modelo customizado



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
import os
import time
import warnings
warnings.filterwarnings("ignore")

# Ultralytics / YOLOv8
from ultralytics import YOLO

# Configuração de device
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
import ultralytics
print(f"Versao do Yolo: {ultralytics.__version__}")


In [ ]:
# Detecção customizada com YOLOv8 em imagem estática

def detectar_yolo_custom(image_path, model, label=0, limiar_confianca=0.4):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inicio = time.time()
    resultados = model(
        img_rgb,
        conf=limiar_confianca,
        classes=[label],        # classe 0 = pessoa no dataset COCO
        verbose=False
    )
    inferencia_ms = (time.time() - inicio) * 1000  # em ms
    retangulos = resultados[0].boxes
    deteccoes = []
    if retangulos is not None:
        for retang in retangulos:
            x1, y1, x2, y2 = retang.xyxy[0].cpu().numpy().astype(int)
            conf = float(retang.conf[0])
            deteccoes.append({"retang": (x1, y1, x2, y2), "conf": conf})
    
    return {
        "img": img_rgb,
        "label": label,
        "deteccoes": deteccoes,
        "inferencia_ms": inferencia_ms,
        "num_deteccoes": len(deteccoes)
    }


def visualizar_deteccoes_yolo(result, title="YOLOv8 customizado"):
    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    label_name = yolo_custom.names[result["label"]]
    ax.imshow(result["img"])
    cores = plt.cm.Set1(np.linspace(0, 1, max(len(result["deteccoes"]), 1)))
    for i, det in enumerate(result["deteccoes"]):
        x1, y1, x2, y2 = det["retang"]
        conf = det["conf"]
        color = cores[i % len(cores)]
        # retangulo deteccao
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2.5, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        # rótulo com confiança
        ax.text(
            x1, y1 - 6, f"{label_name.title()} {conf:.2f}",
            color="white", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8)
        )
    
    ax.set_title(
        f"{title}\n"
        f"{result['num_deteccoes']} {label_name}(s) detectada(s) | "
        f"Inferência: {result['inferencia_ms']:.1f}ms",
        fontsize=10, pad=12
    )
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("output/yolo_deteccoes.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Salvo em output/yolo_deteccoes.png")


In [ ]:
print("Carregando modelo customizado...")

# ------------
# projeto = "dozero_v1_ep200"
# projeto = "dozero_v1_ep30"
# nome_modelo = "yolo_dozero"
# ------------
# projeto = "transfer_v1_ep100"
projeto = "transfer_v1_ep30"
nome_modelo = "yolo_transfer"
# ------------

yolo_custom = YOLO(f'runs/detect/{projeto}/{nome_modelo}/weights/best.pt')

print(60*'-')
print(f"Classes disponíveis:")
for id, name in yolo_custom.names.items():
    print(f"   ID {id}: {name}")
    
print(60*'-')
print(f"   Total de classes: {len(yolo_custom.names)}")


In [ ]:
img_teste = "data/t8.jpg"
# img_teste = "data/imagem_002.png"
result = detectar_yolo_custom(img_teste, yolo_custom, label=1, limiar_confianca=0.2)
print(f"Resultado:")
print(f"   {yolo_custom.names[result['label']].title()}(s) detectadas: {result['num_deteccoes']}")
print(f"   Tempo de inferência: {result['inferencia_ms']:.1f}ms")
for i, d in enumerate(result["deteccoes"]):
    print(f"   [{i+1}] retang={d['retang']} | confiança={d['conf']:.3f}")
visualizar_deteccoes_yolo(result)